# 03 - Erro com pontos flutuantes
Vamos aprender sobre como usar os erros de ponto flutuante para resolução de problemas numéricos.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana3
```

Faça o checkout nessa nova branch:

```bash
git checkout semana3
```

<hr />

## Atividade 1
A função exponencial natural pode ser definida pelo limite:
$$
e^x=\lim_{n\to\infty}\left(1+\frac{x}{n}\right)^n,
$$
mas também é dada pela **série de Maclaurin**:
$$
e^x=\sum_{n=0}^{\infty}\frac{x^n}{n!}
=1+\frac{x}{1!}+\frac{x^2}{2!}+\frac{x^3}{3!}+\cdots
$$

Implemente em **Python** o cálculo de $e^x$ pela série, interrompendo a soma quando o termo ficar menor que o limite prático de contribuição, usando a precisão de máquina como critério.

In [18]:
import math
import sys

def exp_series(x, atol=0.0):
    """ Aproxima e^x pela série de Maclaurin com critério de parada numérico. """
    eps = sys.float_info.epsilon
    s = 1.0
    term = 1.0
    n = 0
    tol_abs = max(atol, eps)

    while True:
        n += 1
        term *= x / n
        s += term
        if abs(term) < eps * abs(s) or abs(term) < tol_abs:
            break
        if n > 10_000:
            break
    return s, n, term

# Demonstração
for val in [1.0, 5.0, -2.0]:
    approx, nterms, last = exp_series(val)
    print(f"x={val:+g} -> e^x ≈ {approx:.16g} (math.exp={math.exp(val):.16g}, termos={nterms})")

x=+1 -> e^x ≈ 2.718281828459046 (math.exp=2.718281828459045, termos=18)
x=+5 -> e^x ≈ 148.4131591025766 (math.exp=148.4131591025766, termos=33)
x=-2 -> e^x ≈ 0.1353352832366127 (math.exp=0.1353352832366127, termos=24)


## Atividade 2

Implemente:
$$
e^x\approx\left(1+\frac{x}{n}\right)^n
$$
com $n$ crescente, e:
1. Explique por que, para $x<0$ e $n$ muito grande, pode ocorrer **cancelamento catastrófico**;
2. Proponha um critério de parada numérico para encerrar o crescimento de $n$ sem perder precisão.

In [19]:
import math
import sys


def exp_limit(x, n0=None, max_iter=60):
    """Aproxima e^x por (1 + x/n)^n."""
    if x == 0:
        return 1.0, 1, 0.0

    eps = sys.float_info.epsilon
    n = max(1, math.floor(abs(x)) + 1) if n0 is None else max(1, n0)
    previsao = None
    change = math.inf

    for _ in range(max_iter):
        correction = x / n
        if abs(correction) <= math.sqrt(eps):
            break

        aproximacao = (1.0 + correction) ** n
        if previsao is not None:
            change = abs(aproximacao - previsao)
            scale = max(abs(aproximacao), abs(previsao), 1.0)
            if change <= 4 * eps * scale:
                break

        previous = aproximacao
        n *= 2

    return previous, n // 2, change


for x in [1.0, -1.0, -2.0, 5.0]:
    aproximacao, n, change = exp_limit(x)
    exact = math.exp(x)
    erro_relativo = abs(aproximacao - exact) / abs(exact)
    print(
        f"x={x:+g}, n={n}: aproximação={aproximacao:.16g}, "
        f"erro relativo={erro_relativo:.3e}"
    )

x=+1, n=33554432: aproximação=2.718281787953491, erro relativo=1.490e-08
x=-1, n=33554432: aproximação=0.3678794356896114, erro relativo=1.490e-08
x=-2, n=100663296: aproximação=0.135335280043579, erro relativo=2.359e-08
x=+5, n=201326592: aproximação=148.4131520994036, erro relativo=4.719e-08


### Resposta

Para `x < 0`, calculamos `1 + x/n` subtraindo dois números próximos. Quando `n` cresce demais, essa subtração perde dígitos significativos: ocorre cancelamento catastrófico. A potência pode então amplificar o erro.

O cálculo para quando `|x|/n <= sqrt(epsilon)`, pois a correção já é pequena e o arredondamento da base começa a dominar. Também verificamos se duas aproximações consecutivas diferem apenas pela precisão de máquina.

## Atividade 3

Para $|x|$ grande, use:
$$
e^x = \left(e^{m\cdot 2^{-k}}\right)^{2^k}, \quad
k = \left\lceil \log_2\!\left(\frac{|x|}{\theta}\right)\right\rceil, \quad m = \frac{x}{2^k}
$$
Calcule $e^{m}$ pela série (Ex. 1) e depois eleve ao quadrado $k$ vezes.

In [20]:
import math


def exp_scaling_squaring(x, theta=1.0):
    if x == 0:
        return 1.0, 0, 0

    k = max(0, math.ceil(math.log2(abs(x) / theta)))
    m = x / (2 ** k)
    resultado, terms, _ = exp_series(m)

    for _ in range(k):
        resultado *= resultado

    return resultado, k, terms


for x in [20.0, 40.0, 50.0]:
    aproximacao, k, terms = exp_scaling_squaring(x)
    exato = math.exp(x)
    erro_relativo = abs(aproximacao - exato) / abs(exato)
    print(
        f"x={x:g}, k={k}, termos={terms}: "
        f"aproximação={aproximacao:.16g}, erro relativo={erro_relativo:.3e}"
    )

x=20, k=5, termos=16: aproximação=485165195.4097913, erro relativo=2.089e-15
x=40, k=6, termos=16: aproximação=2.35385266837021e+17, erro relativo=4.078e-15
x=50, k=6, termos=17: aproximação=5.184705528587007e+21, erro relativo=1.254e-14


## Atividade 4

Use:
$$
\cos x=\sum_{n=0}^{\infty}(-1)^n\frac{x^{2n}}{(2n)!}
$$
com a recursão:
$$
t_{n+1}=t_n\cdot\frac{-x^2}{(2n+1)(2n+2)}
$$
Defina um critério de parada baseado em `epsilon` e compare o erro relativo para $x\in[-20,20]$ (200 pontos) contra `math.cos(x)`.

In [21]:
import math
import sys


def cos_series(x, eps=None):
    """Aproxima cos(x) via série de Maclaurin com recursão."""
    if eps is None:
        eps = sys.float_info.epsilon

    term = 1.0
    s = term
    n = 0

    while True:
        term *= -(x * x) / ((2 * n + 1) * (2 * n + 2))
        s += term
        if abs(term) <= eps * abs(s):
            break
        n += 1

    return s


xs = [(-20.0 + 40.0 * i / 199.0) for i in range(200)]
erro_relativo = []

for x in xs:
    aproximacao = cos_series(x)
    exato = math.cos(x)
    erro_relativo.append(abs(aproximacao - exato) / max(abs(exato), sys.float_info.epsilon))

print(f"Maior erro relativo em [-20, 20]: {max(erro_relativo):.3e}")
for x in [-20.0, -10.0, 0.0, 10.0, 20.0]:
    aproximacao = cos_series(x)
    exato = math.cos(x)
    erro_relativo = abs(aproximacao - exato) / max(abs(exato), sys.float_info.epsilon)
    print(f"x={x:>5.1f}: cos≈{aproximacao:.15f}, erro_relativo={erro_relativo:.3e}")



Maior erro relativo em [-20, 20]: 1.087e-08
x=-20.0: cos≈0.408082062383706, erro_relativo=1.398e-09
x=-10.0: cos≈-0.839071529076605, erro_relativo=1.815e-13
x=  0.0: cos≈1.000000000000000, erro_relativo=0.000e+00
x= 10.0: cos≈-0.839071529076605, erro_relativo=1.815e-13
x= 20.0: cos≈0.408082062383706, erro_relativo=1.398e-09


## Atividade 5

Dado $x$ e uma tolerância $\tau$, encontre o menor $N$ tal que:
$$
R_{N+1}(x)=\sum_{n=N+1}^{\infty}\frac{|x|^n}{n!} < \tau
$$

In [28]:
import math


def menor_N_para_tolerancia(x, tau):
    """Encontra o menor N tal que a cauda da serie de e^{|x|} fica menor que tau."""
    if tau <= 0:
        raise ValueError("tau deve ser positiva.")

    x_abs = abs(x)
    if x_abs == 0:
        return 0

    termo = 1.0
    n = 0

    while True:
        proximo_termo = termo * x_abs / (n + 1)
        termos_cauda = []
        indice = n + 1
        while proximo_termo != 0.0:
            termos_cauda.append(proximo_termo)
            proximo_termo *= x_abs / (indice + 1)
            indice += 1

        resto = math.fsum(termos_cauda)
        if resto < tau:
            return n

        n += 1
        termo *= x_abs / n
        if termo == 0.0 or not math.isfinite(termo):
            raise OverflowError("Nao foi possivel determinar N com float64.")


for x, tau in [(1.0, 1e-8), (5.0, 1e-10), (10.0, 1e-12)]:
    N = menor_N_para_tolerancia(x, tau)
    print(f"x={x}, tau={tau:.1e} -> N={N}")

x=1.0, tau=1.0e-08 -> N=11
x=5.0, tau=1.0e-10 -> N=28
x=10.0, tau=1.0e-12 -> N=46


## Atividade 6

Usando `decimal` ou `mpmath`, compute $e^x$ em alta precisão e compare com o resultado de `float64` (Ex. 1) para $x\in\{20, 40, 50\}$.
Analise:
- perda de dígitos significativos;
- quando o `float64` começa a saturar por overflow.


In [25]:
import math
from decimal import Decimal, getcontext

# alta precisão
getcontext().prec = 80


def exp_float64(x):
    return math.exp(x)


def exp_decimal(x):
    return Decimal(x).exp()


for x in [20.0, 40.0, 50.0]:
    fp = exp_float64(x)
    dp = float(exp_decimal(x))
    rel_err = abs(fp - dp) / max(abs(dp), 1.0)
    print(f"x={x}: float64={fp:.17g}, decimal={dp:.17g}, erro_relativo={rel_err:.3e}")

print("\nLimites do float64:")
for v in [700.0, 710.0, 709.782712893384, 1000.0]:
    try:
        print(f"exp({v}) = {math.exp(v):.3e}")
    except OverflowError:
        print(f"exp({v}) -> overflow")

x=20.0: float64=485165195.40979028, decimal=485165195.40979028, erro_relativo=0.000e+00
x=40.0: float64=2.3538526683702e+17, decimal=2.3538526683702e+17, erro_relativo=0.000e+00
x=50.0: float64=5.184705528587072e+21, decimal=5.184705528587072e+21, erro_relativo=0.000e+00

Limites do float64:
exp(700.0) = 1.014e+304
exp(710.0) -> overflow
exp(709.782712893384) = 1.798e+308
exp(1000.0) -> overflow


## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 3"
git push origin semana3
```